# Получаем данные

In [ ]:
import gdown
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as patches
import textwrap
from datetime import datetime

In [ ]:
df = pd.read_excel("/content/data_1_2_well_cleanedv2.xlsx")
df.head()

In [ ]:
desc_df = pd.read_excel("/content/thesis_burnout_sample_answers3.xlsx")

def get_level(percentile):
    if percentile < 25:
        return "low"
    elif percentile < 75:
        return "middle"
    else:
        return "high"

desc_df

In [ ]:
grouping_cols = ['Регион', 'employment']

In [ ]:
value_dict = {
    "Уровень истощения": "#C8F28D",
    "Внутреннее дистанцирование": "#8DE6F2",
    "Когнитивные затруднения": "#8DC8F2",
    "Эмоциональные затруднения": "#C8F28D"

}

value_list = list(value_dict.keys())

In [ ]:
#колокнки для рекомендаций
def rename_future_columns(df):
    rename_dict = {
        "burnout1_exh": "Уровень истощения",
        "burnout1_dist": "Внутреннее дистанцирование",
        "burnout1_cog": "Когнитивные затруднения",
        "burnout1_emt": "Эмоциональные затруднения"
    }

    return df.rename(columns=rename_dict)


df = rename_future_columns(df)
df.head(2)

In [ ]:
cols = ["id",
        "Уровень истощения",
        "Внутреннее дистанцирование",
        "Когнитивные затруднения",
        "Эмоциональные затруднения"
        ]

cols += grouping_cols

In [ ]:
df = df[cols].copy()
df.head()

# Считаю перцентили

In [ ]:
df["id"] = df["id"].fillna(0).astype(int)

In [ ]:
for col in value_list:
    new_col = f"{col}_emp_perc"
    df[new_col] = (df.groupby(grouping_cols)[col].rank(pct=True) * 100).round(0).astype(int)

df.head()

In [ ]:
# достаем данные для кого-то конкретного ответчика
responder = df.loc[df['id'] == 80623773]

if not responder.empty:
    row_dict = responder.iloc[0].to_dict()
    print(row_dict)

{'id': 80623773, 'Уровень истощения': 2.0, 'Внутреннее дистанцирование': 1.666666666666667, 'Когнитивные затруднения': 1.0, 'Эмоциональные затруднения': 1.0, 'Регион': 1, 'employment': 'Врач', 'Уровень истощения_emp_perc': 13, 'Внутреннее дистанцирование_emp_perc': 41, 'Когнитивные затруднения_emp_perc': 12, 'Эмоциональные затруднения_emp_perc': 12}


In [ ]:

VALUE_COLORS = {
    "Когнитивные затруднения": "#C8F28D",
    "Внутреннее дистанцирование": "#8DE6F2",
    "Эмоциональные затруднения": "#8DC8F2",
    "Уровень истощения": "#F28DA1"
}

COLORS = {
    'accent': '#6C5B7B',
    'text_main': '#2C3E50',
    'low': '#F4AFA9',
    'middle': '#F9D9A0',
    'high': '#9FD7B2'
}

In [ ]:
def save_styled_burnout_report(row_dict, desc_df, filename='burnout_report_2pages4.pdf'):

    emotional_scales = []
    recommendations_rows = [["Цель", "Рекомендация"]]
    scales_to_process = ["Когнитивные затруднения", "Внутреннее дистанцирование",
                         "Эмоциональные затруднения", "Уровень истощения"]

    for scale_name in scales_to_process:
        perc_key = f"{scale_name}_emp_perc"
        if perc_key in row_dict:
            val = row_dict[perc_key]
            lvl = get_level(val)

            match = desc_df[(desc_df['scale'] == scale_name) & (desc_df['level'] == lvl)]

            if not match.empty:
                emotional_scales.append({
                    "label": scale_name,
                    "label_color": VALUE_COLORS.get(scale_name, "#CCCCCC"),
                    "description": match['description'].values[0],
                    "value": val
                })
                recommendations_rows.append([match['aims'].values[0], match['recommendations'].values[0]])

    with PdfPages(filename) as pdf:

        # ГРАФИКИ И ОПИСАНИЯ
        fig1 = plt.figure(figsize=(8.27, 11.69))

        ax_h1 = fig1.add_axes([0, 0.92, 1, 0.08])
        ax_h1.axis('off')
        ax_h1.add_patch(patches.Rectangle((0, 0), 1, 1, color=COLORS['accent']))
        ax_h1.text(0.05, 0.5, "РЕЗУЛЬТАТЫ ОПРОСА (ЛИСТ 1/2)", color='white', fontsize=16, fontweight='bold', va='center')
        ax_h1.text(0.95, 0.5, datetime.now().strftime("%d.%m.%Y"), color='white', fontsize=10, ha='right', va='center')

        # График
        ax_chart = fig1.add_axes([0.38, 0.76, 0.52, 0.12])
        labels = [textwrap.fill(s['label'], 20) for s in emotional_scales]
        values = [s['value'] for s in emotional_scales]
        ax_chart.barh(labels, values, color=[s['label_color'] for s in emotional_scales], height=0.6)
        ax_chart.set_xlim(0, 100)
        ax_chart.spines[['top', 'right', 'bottom']].set_visible(False)
        for i, v in enumerate(values):
            ax_chart.text(v + 1, i, f'{v}%', va='center', fontweight='bold', fontsize=9)

        #  Индикаторы
        start_y = 0.63
        for i, scale in enumerate(emotional_scales):
            y_pos = start_y - (i * 0.14)
            fig1.text(0.1, y_pos + 0.04, scale['label'].upper(), fontsize=10, fontweight='bold', color=scale['label_color'])
            fig1.text(0.1, y_pos + 0.02, textwrap.fill(scale['description'], width=65), fontsize=9, va='top', color=COLORS['text_main'], linespacing=1.4)

            ax_ind = fig1.add_axes([0.70, y_pos - 0.01, 0.28, 0.015])
            ax_ind.axis('off')
            for j, c in enumerate([COLORS['high'], COLORS['middle'], COLORS['low']]):
                ax_ind.add_patch(patches.Rectangle((j*0.25 if j==0 else (0.25 if j==1 else 0.75), 0), (0.25 if j==0 or j==2 else 0.5), 1, color=c))

            val_norm = scale['value'] / 100
            ax_ind.plot([val_norm, val_norm], [-0.5, 1.5], color='black', lw=2)
            ax_ind.text(val_norm, 2.5, str(scale['value']), ha='center', fontsize=8, fontweight='bold')

        pdf.savefig(fig1)
        plt.close(fig1)

        #  РЕКОМЕНДАЦИИ
        fig2 = plt.figure(figsize=(8.27, 11.69))

        ax_h2 = fig2.add_axes([0, 0.92, 1, 0.08])
        ax_h2.axis('off')
        ax_h2.add_patch(patches.Rectangle((0, 0), 1, 1, color=COLORS['accent']))
        ax_h2.text(0.05, 0.5, "ИНДИВИДУАЛЬНЫЕ РЕКОМЕНДАЦИИ (ЛИСТ 2/2)", color='white', fontsize=16, fontweight='bold', va='center')

        # Большая таблица
        ax_table = fig2.add_axes([0.1, 0.1, 0.8, 0.75])
        ax_table.axis('off')

        wrapped_table_data = []
        for row_idx, row in enumerate(recommendations_rows):
            if row_idx == 0:
                wrapped_table_data.append(row)
            else:
                new_row = [
                    textwrap.fill(str(row[0]), width=19),
                    textwrap.fill(str(row[1]), width=55)
                ]
                wrapped_table_data.append(new_row)

        table = ax_table.table(
            cellText=wrapped_table_data[1:],
            colLabels=wrapped_table_data[0],
            loc='upper center',
            cellLoc='left',
            colWidths=[0.3, 0.7]
        )

        table.auto_set_font_size(False)
        table.set_fontsize(10)

        #(ширина, высота).
        table.scale(1.1, 4)

        for (row, col), cell in table.get_celld().items():
            cell.set_edgecolor('#DDDDDD')
            if row == 0:
                cell.set_facecolor(COLORS['accent'])
                cell.set_text_props(color='white', weight='bold', ha='center')
            else:
                cell.set_text_props(ha='left', va='center')

        pdf.savefig(fig2)
        plt.close(fig2)

    print(f"отчет сохранен: {filename}")

save_styled_burnout_report(row_dict, desc_df)

# Отправка

In [ ]:
import smtplib
from email.message import EmailMessage


user_email = 'm.tiulpakova@dodobrands.io' #'#['marinalovescake11@gmail.com',

pdf_path = "/content/burnout_report_2pages4.pdf"


sender_email = ...
sender_password =# App Password
subject = "Ваш персональный отчет"
body = "Здравствуйте!\nВаш отчет о выгорании во вложении.\n\n"
body += "\n\nС уважением,\nКоманда проекта"

msg = EmailMessage()
msg["From"] = sender_email
msg["To"] = user_email
msg["Subject"] = subject
msg.set_content(body)

with open(pdf_path, "rb") as f:
    file_data = f.read()
    msg.add_attachment(file_data, maintype="application", subtype="pdf", filename=pdf_path.split("/")[-1])


with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
    smtp.login(sender_email, sender_password)
    smtp.send_message(msg)

print(f"Email sent to {user_email}")
